In [ ]:
!pip install dspy

In [ ]:
import os
import dspy
import re
import csv
import shutil
from pathlib import Path
from typing import List, Dict, Set, Optional
from google.colab import drive, ai


# Directory paths
XML_FOLDER = "/content/drive/MyDrive/AI6129/xml"
FORMAT_RULES_FILE = "/content/drive/MyDrive/AI6129/accession_formats.md"


In [ ]:
def setup_colab_environment():
    """
    Setup Google Colab environment.
    - Mounts Google Drive
    - Configures DSPy with Colab's built-in AI (NO API KEY NEEDED)
    """
    print("="*80)
    print("SETTING UP COLAB ENVIRONMENT")
    print("="*80)

    # Mount Google Drive
    print("\n1. Mounting Google Drive...")
    try:
        drive.mount('/content/drive', force_remount=False)
        print("   Google Drive mounted")
    except Exception as e:
        print(f"  Error mounting Drive: {e}")
        return False

    # Check Colab AI availability
    print("\n2. Checking Colab AI availability...")
    try:
        # Test the built-in AI
        test_response = ai.generate_text(
            "Say 'ready'",
            model_name='google/gemini-2.5-flash'
        )
        print("   Colab AI is available (Gemini models)")
        print(f"  Test response: {test_response[:50]}...")
    except Exception as e:
        print(f"  Error accessing Colab AI: {e}")
        print("   Note: Colab AI requires Pro or Pro+ subscription")
        return False

    # Configure DSPy with custom Colab AI wrapper
    print("\n3. Configuring DSPy with Colab AI...")
    try:
        lm = ColabAIWrapper()
        dspy.configure(lm=lm)
        print("  DSPy configured with Colab's built-in Gemini")
    except Exception as e:
        print(f" Error configuring DSPy: {e}")
        return False

    print("\n" + "="*80)
    print("SETUP COMPLETE")
    print("="*80 + "\n")
    return True



# ==================================================
# COLAB AI WRAPPER FOR DSPy
# ==================================================

In [ ]:
class ColabAIWrapper(dspy.LM):
    """
    Wrapper to use Google Colab's built-in AI with DSPy.
    No API key required - uses Colab Pro/Pro+ built-in models.
    """

    def __init__(self, model_name='google/gemini-2.5-flash-lite', **kwargs):
        """
        Initialize Colab AI wrapper.

        Args:
            model_name: Colab AI model to use
                - 'google/gemini-2.0-flash-lite' (default, fast)
                - 'google/gemini-2.0-flash' (more capable)
        """
        self.model_name = model_name
        self.kwargs = kwargs

        # Initialize base LM
        super().__init__(model=model_name)

    def __call__(self, prompt=None, messages=None, **kwargs):
        """
        Make a completion request using Colab AI.

        Args:
            prompt: String prompt (for simple queries)
            messages: List of message dicts (for chat format)
            **kwargs: Additional generation parameters

        Returns:
            List of strings (completions)
        """
        # Combine instance kwargs with call kwargs
        generation_kwargs = {**self.kwargs, **kwargs}

        # Convert messages to prompt if needed
        if messages:
            prompt = self._messages_to_prompt(messages)
        elif not prompt:
            raise ValueError("Either 'prompt' or 'messages' must be provided")

        # Generate using Colab AI
        try:
            stream = ai.generate_text(
                prompt,
                model_name=self.model_name,
                stream=True
            )

            # Collect all chunks into complete response  # changed
            response_parts = []  # changed
            for chunk in stream:  # changed
                response_parts.append(chunk)  # changed

            response = "".join(response_parts)  # changed

            # Return in format expected by DSPy
            return [response]

        except Exception as e:
            print(f"Error in Colab AI generation: {e}")
            raise

    def _messages_to_prompt(self, messages):
        """
        Convert DSPy message format to a single prompt string.

        Args:
            messages: List of message dicts with 'role' and 'content'

        Returns:
            Formatted prompt string
        """
        prompt_parts = []

        for msg in messages:
            role = msg.get('role', 'user')
            content = msg.get('content', '')

            if role == 'system':
                prompt_parts.append(f"System: {content}")
            elif role == 'user':
                prompt_parts.append(f"User: {content}")
            elif role == 'assistant':
                prompt_parts.append(f"Assistant: {content}")

        return "\n\n".join(prompt_parts)

# ===========================================================
# RESOURCE LOADING
# ===========================================================

In [ ]:
def load_format_rules(filepath: str = FORMAT_RULES_FILE) -> Optional[str]:
    """
    Load accession number format rules from markdown file.

    Args:
        filepath: Path to markdown file containing format rules

    Returns:
        String content of format rules, or None if error
    """

    print(f"Loading format rules from: {filepath}")

    if not os.path.exists(filepath):
        print(f" Format rules file not found: {filepath}")
        return None

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
        print(f"Format rules loaded ({len(content)} characters)")
        return content
    except Exception as e:
        print(f"Error reading format file: {e}")
        return None

def load_single_xml(filepath: str) -> Optional[str]:
  """
    Load a single XML file.

    Args:
        filepath: Path to XML file

    Returns:
        XML content as string, or None if error
  """
  if not os.path.exists(filepath):
      print(f" XML file not found: {filepath}")
      return None

  try:
      with open(filepath, 'r', encoding='utf-8') as f:
          content = f.read()

      size_kb = len(content) / 1024
      print(f" Loaded XML ({size_kb:.1f} KB)")
      return content

  except Exception as e:
      print(f" Error reading XML file: {e}")
      return None

def load_xml_folder(folder_path: str = XML_FOLDER, max_files: Optional[int] = None) -> Dict[str, str]:
  """
  Load all XML files from a folder.

  Args:
      folder_path: Path to folder containing XML files
      max_files: Optional limit on number of files to load

  Returns:
      Dictionary mapping filename to XML content
  """

  xml_files = {}

  if not os.path.exists(folder_path):
      print(f" Folder not found: {folder_path}")
      return xml_files

  xml_file_list = list(Path(folder_path).glob("*.xml"))
  if not xml_file_list:
    print(f" No XML files found in: {folder_path}")
    return xml_files

  print(f"Found {len(xml_file_list)} XML files in folder")

  if max_files:
    xml_file_list = xml_file_list[:max_files]
    print(f" Loading only the first {max_files} files")

  for xml_file in xml_file_list:
    try:
      with open(xml_file, 'r', encoding='utf-8') as f:
        content = f.read()

      xml_files[xml_file.name] = content
      size_kb = len(content) / 1024
      print(f" {xml_file.name} ({size_kb:.1f} KB)")

    except Exception as e:
      print(f" Error reading XML file: {e}")

  print(f"Successfully loaded {len(xml_files)} files")
  return xml_files


# ==========================================================
# DSPy SIGNATURE & MODULE
# ==========================================================

In [ ]:
class AccessionExtractionSignature(dspy.Signature):
    """Extract accession identifiers that appear verbatim in the article."""

    format_rules: str = dspy.InputField(
        desc=(
            "Markdown reference for valid accession formats and known false positives. "
            "Use it ONLY to validate candidates; do not copy examples from it."
        )
    )

    xml_content: str = dspy.InputField(
        desc="Full article XML to extract from."
    )

    accessions: list[str] = dspy.OutputField(
        desc=(
            "Unique accession strings found in xml_content (verbatim). "
            "Return [] if none. Do not include uncertain candidates."
        )
    )

    notes: list[str] = dspy.OutputField(
        desc="Optional brief notes about exclusions/uncertainty; keep short."
    )

# class AccessionExtractionSignature(dspy.Signature):  #changed - entire docstring rewritten
#     """
#     Extract database accession numbers from scientific article XML.

#     SCOPE OF EXTRACTION:
#     Extract ALL sequence database accession numbers, including but not limited to:
#     - GenBank/INSDC accessions (e.g., MN908947, AAWZUW000000000)
#     - SRA/Sequence Read Archive accessions (e.g., SRR12967557, SRP, SRS, SRX prefixes)
#     - BioProject accessions (e.g., PRJNA123456)
#     - BioSample accessions (e.g., SAMN12345678)
#     - RefSeq accessions (e.g., NM_001234, NC_045512)
#     - DDBJ accessions
#     - ENA/European Nucleotide Archive accessions
#     - Whole Genome Shotgun (WGS) accessions
#     - Transcriptome Shotgun Assembly (TSA) accessions

#     DECISION FRAMEWORK:
#     1. Scan the article for patterns matching format_rules
#     2. For each candidate, verify it EXACTLY appears in xml_content
#     3. Confirm it is NOT a format example, DOI, or other identifier
#     4. If uncertain about ANY candidate, DO NOT include it
#     5. Return 'NONE FOUND' if no candidates pass all checks

#     WHAT TO EXTRACT (only if ALL criteria met):
#     - Database accession numbers matching patterns in format_rules
#     - Each accession ONCE only (no duplicates)
#     - Only accessions that EXACTLY appear in the xml_content

#     WHAT TO EXCLUDE (never extract these):
#     - Format examples from format_rules (e.g., AB123456, XM_123456.1)
#     - DOI components (contain hyphens like 10.1038/s41588-021-00978-w)
#     - Journal identifiers (e.g., s41588-021-00978-w, pone.0123456)
#     - Reference numbers in brackets (e.g., [1], [2], [S1])
#     - Gene names (e.g., BRCA1, TP53) or protein names
#     - Sample identifiers created by authors (e.g., HAV-001, Sample_A)
#     - Figure or table references (e.g., Fig1, Table2)
#     - Patent numbers or grant identifiers
#     - ATCC/VR catalogue numbers (e.g., VR-5, VR-1775) - these are NOT GenBank

#     CALIBRATION PRINCIPLE:
#     It is BETTER to return 'NONE FOUND' than to include uncertain candidates.
#     Precision is more important than recall for this task.
#     """

#     format_rules: str = dspy.InputField(
#         desc="Reference document defining valid database accession number formats. "
#              "Use ONLY to validate patterns, NOT as a source of accessions to extract. "
#              "All examples in this document (e.g., AB123456) are illustrative only."
#     )

#     xml_content: str = dspy.InputField(
#         desc="Full XML content of the scientific article. "
#              "May or may not contain GenBank accession numbers. "
#              "Extract only accessions that explicitly appear in this content."
#     )

#     accessions: str = dspy.OutputField(
#         desc="If valid accessions found: 'ACCESSION' per line. "
#              "If NO valid accessions found: return exactly 'NONE FOUND'. "
#              "Each accession must exactly match a pattern from format_rules "
#              "AND explicitly appear in the xml_content. "
#              "When in doubt, exclude the candidate and return 'NONE FOUND'."
#     )


class AccessionExtractor(dspy.Module):
    """
    DSPy module for accession extraction with chain-of-thought reasoning.

    Uses ChainOfThought to encourage step-by-step reasoning:
        1. Assess whether article likely contains accessions (presence check)
        2. Review format rules to understand valid accession patterns
        3. Scan article for candidate patterns
        4. Validate each candidate against exclusion criteria
        5. Return formatted results or 'NONE FOUND'
    """

    def __init__(self):
        super().__init__()
        self.extract = dspy.ChainOfThought(AccessionExtractionSignature)

    def forward(self, format_rules: str, xml_content: str):
        """
        Extract accessions using Chain of Thought reasoning.

        Args:
            format_rules: GenBank format rules
            xml_content: Article XML content

        Returns:
            DSPy prediction with accessions field
        """
        return self.extract(format_rules=format_rules, xml_content=xml_content)



# =========================================================
# EXTRACTION FUNCTIONS
# =========================================================

In [ ]:
def extract_from_single_file(xml_filepath: str, format_rules_path: str = FORMAT_RULES_FILE) -> Dict:
    """
    Extract accession numbers from a single XML file.

    Args:
        xml_filepath: Path to XML file
        format_rules_path: Path to format rules markdown

    Returns:
        Dictionary with extraction results
    """

    print("\n" + "="*80)
    print("EXTRACTING FROM SINGLE FILE")
    print("="*80 + "\n")

    #load format rules
    format_rules = load_format_rules(format_rules_path)
    if not format_rules:
      return {"error": "Failed to load format rules"}

    #Load XML file
    print(f"\nLoading XML: {xml_filepath}")
    xml_content = load_single_xml(xml_filepath)
    if not xml_content:
      return {"error": "Failed to load XML file"}

    #Extract accessions
    print("\nExtracting accession numbers using Colab AI...")
    extractor = AccessionExtractor()

    try:

        result = extractor(format_rules=format_rules, xml_content=xml_content)

        print("\n" + "-"*80)
        print("EXTRACTION RESULTS")
        print("-"*80)
        print(result.accessions)
        print("-"*80 + "\n")
        print("DSPY REASONING")
        print(result.reasoning)

        return {
            "filename": os.path.basename(xml_filepath),
            "accessions": result.accessions,
            "raw_result": result
        }

    except Exception as e:
        print(f" Extraction error: {e}")
        return {"error": str(e)}

def extract_from_folder(folder_path: str = XML_FOLDER, format_rules_path: str = FORMAT_RULES_FILE, max_files: Optional[int] = None) -> List[Dict]:
    """
    Extract accession numbers from all XML files in a folder.

    Args:
        folder_path: Path to folder containing XML files
        format_rules_path: Path to format rules markdown
        max_files: Optional limit on number of files to process

    Returns:
        List of dictionaries with extraction results per file
    """

    print("\n" + "="*80)
    print("EXTRACTING FROM FOLDER")
    print("="*80 + "\n")

    #Load format rules
    format_rules = load_format_rules(format_rules_path)
    if not format_rules:
      return {"error": "Failed to load format rules"}

    #Load XML files
    print(f"\nLoading XML files from: {folder_path}")
    xml_files = load_xml_folder(folder_path, max_files)

    if not xml_files:
      return [{"error": "No XML files loaded"}]

    #Extract from each file
    results = []
    extractor = AccessionExtractor()

    for idx, (filename, xml_content) in enumerate(xml_files.items(), 1):
        print(f"\n{'='*80}")
        print(f"Processing [{idx}/{len(xml_files)}]: {filename}")
        print(f"{'='*80}")

        try:
          result = extractor(format_rules=format_rules, xml_content=xml_content)

          print("\n" + "-"*40)
          print("RESULTS:")
          print("-"*40)
          print(result.accessions)
          print("-"*40 + "\n")

          results.append({
              "filename": filename,
              accessions: result.accessions,
              "result": result
          })
        except exception as e:
            print(f" Error processing {filename}: {e}")
            results.append({
                "filename": filename,
                "error": str(e)
            })

    print("\n" + "="*80)
    print(f"EXTRACTION COMPLETE - Processed {len(results)} files")
    print("="*80 + "\n")

    return results

Extract One specific file

In [ ]:
def example_single_file():
    """
    Example: Extract from one specific XML file.
    """
    print("\n" + "#"*80)
    print("EXAMPLE: SINGLE FILE EXTRACTION")
    print("#"*80 + "\n")

    # Specify your file path
    xml_file = "/content/drive/MyDrive/AI6129/xml/PMC9387262_20251201.xml"

    result = extract_from_single_file(xml_file)
    return result

In [ ]:
#PMC8101457_20251201.xml #hallucinaation issue 0 accession
if __name__ == "__main__":
  setup_colab_environment()
  example_single_file()

SETTING UP COLAB ENVIRONMENT

1. Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   Google Drive mounted

2. Checking Colab AI availability...
   Colab AI is available (Gemini models)
  Test response: ready...

3. Configuring DSPy with Colab AI...
  DSPy configured with Colab's built-in Gemini

SETUP COMPLETE


################################################################################
EXAMPLE: SINGLE FILE EXTRACTION
################################################################################


EXTRACTING FROM SINGLE FILE

Loading format rules from: /content/drive/MyDrive/AI6129/accession_formats.md
Format rules loaded (2334 characters)

Loading XML: /content/drive/MyDrive/AI6129/xml/PMC8101457_20251201.xml
 Loaded XML (129.6 KB)

Extracting accession numbers using Colab AI...

--------------------------------------------------------------------------------
EXTRACTION RESU

In [ ]:
#PMC9387262_20251201.xml low recall issue actual 556 accessions
if __name__ == "__main__":
  setup_colab_environment()
  example_single_file()

SETTING UP COLAB ENVIRONMENT

1. Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   Google Drive mounted

2. Checking Colab AI availability...
   Colab AI is available (Gemini models)
  Test response: ready...

3. Configuring DSPy with Colab AI...
  DSPy configured with Colab's built-in Gemini

SETUP COMPLETE


################################################################################
EXAMPLE: SINGLE FILE EXTRACTION
################################################################################


EXTRACTING FROM SINGLE FILE

Loading format rules from: /content/drive/MyDrive/AI6129/accession_formats.md
Format rules loaded (2334 characters)

Loading XML: /content/drive/MyDrive/AI6129/xml/PMC9387262_20251201.xml
 Loaded XML (281.0 KB)

Extracting accession numbers using Colab AI...

--------------------------------------------------------------------------------
EXTRACTION RESU

In [ ]:
#PMC7564550_20251201.xml hallucination issue 0 accession
if __name__ == "__main__":
  setup_colab_environment()
  example_single_file()

SETTING UP COLAB ENVIRONMENT

1. Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   Google Drive mounted

2. Checking Colab AI availability...
  Error accessing Colab AI: generate_text() got an unexpected keyword argument 'max_output_token'
   Note: Colab AI requires Pro or Pro+ subscription

################################################################################
EXAMPLE: SINGLE FILE EXTRACTION
################################################################################


EXTRACTING FROM SINGLE FILE

Loading format rules from: /content/drive/MyDrive/AI6129/accession_formats.md
Format rules loaded (14446 characters)

Loading XML: /content/drive/MyDrive/AI6129/xml/PMC7564550_20251201.xml
 Loaded XML (286.0 KB)

Extracting accession numbers using Colab AI...

--------------------------------------------------------------------------------
EXTRACTION RESULTS
--------------

In [ ]:
#PMC8557873_20251201.xml low recall issue actual 4 accessions
if __name__ == "__main__":
  setup_colab_environment()
  example_single_file()

SETTING UP COLAB ENVIRONMENT

1. Mounting Google Drive...
Mounted at /content/drive
   Google Drive mounted

2. Checking Colab AI availability...
   Colab AI is available (Gemini models)
  Test response: ready...

3. Configuring DSPy with Colab AI...
  DSPy configured with Colab's built-in Gemini

SETUP COMPLETE


################################################################################
EXAMPLE: SINGLE FILE EXTRACTION
################################################################################


EXTRACTING FROM SINGLE FILE

Loading format rules from: /content/drive/MyDrive/AI6129/accession_formats.md
Format rules loaded (14446 characters)

Loading XML: /content/drive/MyDrive/AI6129/xml/PMC8557873_20251201.xml
 Loaded XML (120.6 KB)

Extracting accession numbers using Colab AI...

--------------------------------------------------------------------------------
EXTRACTION RESULTS
--------------------------------------------------------------------------------
NC_016854
NC_0112

In [ ]:
def move_gt_issue_files():
    """
    Move JSON files for PMCIDs with GT_Issue status to a subfolder.
    These are documents where extracted accessions > ground truth accessions,
    indicating the ground truth JSON is incomplete.
    """
    # List of PMCIDs identified with GT_Issue status (extracted > gt)
    gt_issue_pmcids = [
        "PMC8101457",   # extracted: 131, gt: 0
        "PMC6817352",   # extracted: 55, gt: 0
        "PMC9251186",   # extracted: 30, gt: 20
        "PMC8913728",   # extracted: 23, gt: 19
        "PMC5794934",   # extracted: 22, gt: 0
        "PMC9739812",   # extracted: 16, gt: 8
        "PMC8568146",   # extracted: 11, gt: 2
        "PMC6118388",   # extracted: 8, gt: 0
        "PMC6703130",   # extracted: 8, gt: 4
        "PMC6813399",   # extracted: 6, gt: 3
        "PMC9002014",   # extracted: 5, gt: 1
        "PMC8557873",   # extracted: 6, gt: 4
        "PMC5985583",   # extracted: 3, gt: 0
        "PMC6378779",   # extracted: 3, gt: 1
        "PMC5047355",   # extracted: 3, gt: 1
        "PMC7825753",   # extracted: 2, gt: 1
        "PMC8000398",   # extracted: 2, gt: 1
        "PMC8208698",   # extracted: 1, gt: 0
        "PMC7564550",   # extracted: 1, gt: 0
        "PMC8512087",   # extracted: 1, gt: 0
        "PMC4748464",   # extracted: 1, gt: 0
    ]

    # Directory paths
    source_dir = Path("/content/drive/MyDrive/AI6129/ground_truth/original")
    dest_dir = Path("/content/drive/MyDrive/AI6129/ground_truth/original/Incorrect_GT")

    # Create destination directory if it doesn't exist
    dest_dir.mkdir(parents=True, exist_ok=True)
    print(f"Destination directory: {dest_dir}")

    # Counters for summary
    moved_count = 0
    not_found_count = 0
    error_count = 0

    print("\n" + "="*80)
    print("MOVING GT_ISSUE JSON FILES")
    print("="*80 + "\n")

    for pmcid in gt_issue_pmcids:
        json_filename = f"{pmcid}.json"
        source_path = source_dir / json_filename
        dest_path = dest_dir / json_filename

        if not source_path.exists():
            print(f"  [NOT FOUND] {json_filename}")
            not_found_count += 1
            continue

        try:
            shutil.move(str(source_path), str(dest_path))
            print(f"  [MOVED] {json_filename}")
            moved_count += 1
        except Exception as e:
            print(f"  [ERROR] {json_filename}: {e}")
            error_count += 1

    # Summary
    print("\n" + "-"*80)
    print("SUMMARY")
    print("-"*80)
    print(f"  Files moved:     {moved_count}")
    print(f"  Files not found: {not_found_count}")
    print(f"  Errors:          {error_count}")
    print(f"  Total PMCIDs:    {len(gt_issue_pmcids)}")

    return {
        "moved": moved_count,
        "not_found": not_found_count,
        "errors": error_count
    }

In [ ]:
# Execute the file move
move_gt_issue_files()

Destination directory: /content/drive/MyDrive/AI6129/ground_truth/original/Incorrect_GT

MOVING GT_ISSUE JSON FILES

  [MOVED] PMC8101457.json
  [MOVED] PMC6817352.json
  [MOVED] PMC9251186.json
  [MOVED] PMC8913728.json
  [MOVED] PMC5794934.json
  [MOVED] PMC9739812.json
  [MOVED] PMC8568146.json
  [MOVED] PMC6118388.json
  [MOVED] PMC6703130.json
  [MOVED] PMC6813399.json
  [MOVED] PMC9002014.json
  [MOVED] PMC8557873.json
  [MOVED] PMC5985583.json
  [MOVED] PMC6378779.json
  [MOVED] PMC5047355.json
  [MOVED] PMC7825753.json
  [MOVED] PMC8000398.json
  [MOVED] PMC8208698.json
  [MOVED] PMC7564550.json
  [MOVED] PMC8512087.json
  [MOVED] PMC4748464.json

--------------------------------------------------------------------------------
SUMMARY
--------------------------------------------------------------------------------
  Files moved:     21
  Files not found: 0
  Errors:          0
  Total PMCIDs:    21


{'moved': 21, 'not_found': 0, 'errors': 0}

## Cell B: Extract Accessions for GT_Issue PMCIDs

Output: `corrected_gt.csv` (pipe-delimited)

In [ ]:


# List of PMCIDs with GT_Issue status
GT_ISSUE_PMCIDS = [
    "PMC7825753", "PMC8208698", "PMC7564550",
    "PMC6118388", "PMC6703130", "PMC9002014",
    "PMC8557873", "PMC6378779", "PMC8568146",
    "PMC8913728", "PMC8101457"
]


def parse_accessions_from_output(raw_output) -> List[str]:
    """
    Parse accession numbers from DSPy extraction output.

    Parameters:
        raw_output: Raw output from AccessionExtractor (can be list or string)

    Returns:
        List of unique accession numbers (deduplicated, uppercase)
    """
    accessions = []

    # Handle None or empty
    if not raw_output:
        return accessions

    # Handle list output (new format)  # v1.1 #changed
    if isinstance(raw_output, list):  # v1.1 #changed
        for item in raw_output:  # v1.1 #changed
            if item is None:  # v1.1 #changed
                continue  # v1.1 #changed
            acc = str(item).strip().upper()  # v1.1 #changed
            if acc and acc != "NONE FOUND" and len(acc) >= 6:  # v1.1 #changed
                accessions.append(acc)  # v1.1 #changed

    # Handle string output (old format)  # v1.1 #changed
    elif isinstance(raw_output, str):  # v1.1 #changed
        # Handle 'NONE FOUND' case
        if "NONE FOUND" in raw_output.upper():
            return accessions

        # Split by newlines and extract accession from each line
        lines = raw_output.strip().split('\n')

        for line in lines:
            line = line.strip()
            if not line:
                continue

            # Extract first element if pipe-delimited
            if '|' in line:
                acc = line.split('|')[0].strip().upper()
            else:
                acc = line.split()[0].upper() if line.split() else ""

            if acc and len(acc) >= 6:
                accessions.append(acc)

    # Remove duplicates while preserving order
    seen = set()
    unique_accessions = []
    for acc in accessions:
        if acc not in seen:
            seen.add(acc)
            unique_accessions.append(acc)

    return unique_accessions

def find_xml_file(pmcid: str, xml_folder: str) -> str:
    """
    Find the XML file for a given PMCID.
    XML files may have date suffixes like PMC8101457_20251201.xml

    Parameters:
        pmcid: The PMCID to find
        xml_folder: Path to folder containing XML files

    Returns:
        Full path to XML file, or empty string if not found
    """
    xml_path = Path(xml_folder)

    # Try exact match first
    exact_match = xml_path / f"{pmcid}.xml"
    if exact_match.exists():
        return str(exact_match)

    # Try pattern match with date suffix
    pattern_matches = list(xml_path.glob(f"{pmcid}_*.xml"))
    if pattern_matches:
        return str(pattern_matches[0])

    return ""

In [ ]:
def extract_accessions_for_gt_issues(
    pmcid_list: List[str],
    xml_folder: str = "/content/drive/MyDrive/AI6129/xml",
    format_rules_path: str = "/content/drive/MyDrive/AI6129/accession_formats.md",
    output_csv: str = "/content/drive/MyDrive/AI6129/accession/audit/corrected_gt.csv"
) -> List[Dict]:
    """
    Extract accessions for PMCIDs with GT issues using DSPy.
    Output results to a pipe-delimited CSV file.

    Parameters:
        pmcid_list: List of PMCIDs to process
        xml_folder: Path to XML files
        format_rules_path: Path to format rules markdown
        output_csv: Path for output CSV file

    Returns:
        List of dictionaries with extraction results
    """
    print("\n" + "="*80)
    print("EXTRACTING ACCESSIONS FOR GT_ISSUE PMCIDs")
    print("="*80 + "\n")

    # Load format rules
    format_rules = load_format_rules(format_rules_path)
    if not format_rules:
        print("ERROR: Failed to load format rules")
        return []

    # Initialise extractor
    extractor = AccessionExtractor()

    results = []

    for idx, pmcid in enumerate(pmcid_list, 1):
        print(f"\n[{idx}/{len(pmcid_list)}] Processing: {pmcid}")
        print("-" * 40)

        # Find XML file
        xml_filepath = find_xml_file(pmcid, xml_folder)
        if not xml_filepath:
            print(f"  WARNING: XML file not found for {pmcid}")
            results.append({
                "pmcid": pmcid,
                "accessions": "",
                "status": "XML_NOT_FOUND"
            })
            continue

        # Load XML content
        xml_content = load_single_xml(xml_filepath)
        if not xml_content:
            print(f"  WARNING: Failed to load XML for {pmcid}")
            results.append({
                "pmcid": pmcid,
                "accessions": "",
                "status": "XML_LOAD_ERROR"
            })
            continue

        # Extract accessions using DSPy
        try:
            result = extractor(format_rules=format_rules, xml_content=xml_content)

            print("DSPY RESULT")
            print("-"*80)
            print(result.accessions)
            print("-"*80 + "\n")
            print("DSPY REASONING")
            print(result.reasoning)

            accession_list = parse_accessions_from_output(result.accessions)
            accessions_str = ",".join(accession_list)

            print(f"  Extracted {len(accession_list)} accessions")

            if accession_list:
                print(f"  Accessions: {accessions_str[:100]}{'...' if len(accessions_str) > 100 else ''}")

            results.append({
                "pmcid": pmcid,
                "accessions": accessions_str,
                "status": "SUCCESS"
            })

        except Exception as e:
            print(f"  ERROR: Extraction failed - {e}")
            results.append({
                "pmcid": pmcid,
                "accessions": "",
                "status": f"ERROR: {str(e)}"
            })

    # Write results to pipe-delimited CSV
    print("\n" + "="*80)
    print(f"WRITING OUTPUT TO: {output_csv}")
    print("="*80 + "\n")

    with open(output_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f, delimiter='|')
        writer.writerow(['pmcid', 'accessions'])  # Header

        for result in results:
            writer.writerow([result['pmcid'], result['accessions']])

    # Summary
    success_count = sum(1 for r in results if r['status'] == 'SUCCESS')
    print(f"Results written: {len(results)} rows")
    print(f"Successful extractions: {success_count}")
    print(f"Output file: {output_csv}")

    return results

In [ ]:
# Execute extraction for GT_Issue PMCIDs
if __name__ == "__main__":
    setup_colab_environment()
    extract_accessions_for_gt_issues(GT_ISSUE_PMCIDS)

SETTING UP COLAB ENVIRONMENT

1. Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   Google Drive mounted

2. Checking Colab AI availability...
   Colab AI is available (Gemini models)
  Test response: Ready...

3. Configuring DSPy with Colab AI...
  DSPy configured with Colab's built-in Gemini

SETUP COMPLETE


EXTRACTING ACCESSIONS FOR GT_ISSUE PMCIDs

Loading format rules from: /content/drive/MyDrive/AI6129/accession_formats.md
Format rules loaded (2334 characters)

[1/11] Processing: PMC7825753
----------------------------------------
 Loaded XML (122.8 KB)
DSPY RESULT
--------------------------------------------------------------------------------
[]
--------------------------------------------------------------------------------

DSPY REASONING
The document was scanned for accession numbers. No strings that match the provided accession format rules were found in the text. Ther